# Player Analysis - Batch Processing - Gold Layer

At the Gold layer, data is primed and refined to answer the below questions:

1. Goal based Best Team, Efficient Team and High Scoring Teams vs Defensive Teams
2. Disciplined vs Most Aggressive Teams based on bookings
3. Most Tactical Teams (High Substitutions) and Substitution Timing(early/late)
4. Classify Teams grade using Penalty

In [0]:
# And uninstall a few helpers we prepared for you
%pip uninstall -y databricks_helpers exercise_ev_databricks_unit_tests

# now install the databricks helpers 1
%pip install git+https://github.com/data-derp/databricks_helpers.git

# # now install and also the databricks test cases
%pip install git+https://github.com/data-derp/exercise_ev_databricks_unit_tests.git

In [0]:
exercise_name = "team_analysis_batch_processing/gold"
from pyspark.sql import DataFrame
from databricks_helpers.databricks_helpers import DataDerpDatabricksHelpers
from pyspark.sql.functions import col

helpers = DataDerpDatabricksHelpers(dbutils, exercise_name)

current_user = helpers.current_user()
working_directory = helpers.working_directory()
helpers.clean_working_directory()

silver_working_directory = working_directory.replace("gold", "silver")
bronze_working_directory = working_directory.replace("gold", "bronze")
silver_output = f"{silver_working_directory}/output"
bronze_output = f"{bronze_working_directory}/output"

In [0]:
goals_silver_df = spark.read.parquet(f"{silver_output}/goals_per_team")
bookings_silver_df = spark.read.parquet(f"{silver_output}/bookings_by_team")
subs_silver_df = spark.read.parquet(f"{silver_output}/substitutions_by_team")
penalty_kicks_silver_df = spark.read.parquet(f"{silver_output}/penalty_kicks_by_team")


### **1. Goal based Best Team, Efficient Team and High Scoring Teams vs Defensive Teams**

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

best_goals_gold = goals_silver_df.groupBy(
    "team_name",
    "team_code"
).agg(
    F.count("*").alias("total_goals"),
    # F.countDistinct("match_id").alias("matches_played"),
    # F.sum("is_early_goal").alias("early_goals"),
    # F.sum(F.when(F.col("penalty") == 1, 1).otherwise(0)).alias("penalty_goals"),
    # F.sum(F.when(F.col("own_goal") == 1, 1).otherwise(0)).alias("own_goals")
)

# Performance metric
# best_goals_gold = best_goals_gold.withColumn(
#     "goals_per_match",
#     F.round(F.col("total_goals") / F.col("matches_played"), 2)
# )

# Ranking
window_spec = Window.orderBy(F.desc("total_goals"))

best_goals_gold = best_goals_gold.withColumn(
    "rank",
    F.dense_rank().over(window_spec)
).withColumn(
    "is_top_team",
    F.when(F.col("rank") == 1, "Yes").otherwise("No")
)

display(best_goals_gold.orderBy("rank").limit(10))

In [0]:
def write(input_df: DataFrame):
    out_dir = f"{working_directory}/output/best_goals_gold"
    
### Put your code here.
    mode_name = "overwrite"
    
    input_df. \
        write. \
        mode(mode_name). \
        parquet(out_dir)
    
    
write(best_goals_gold)
dbutils.fs.ls(f"{working_directory}/output/best_goals_gold")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# STEP 1: Goals per team per year
year_team_goals = goals_silver_df.groupBy(
    "year",
    "team_name",
    "team_code"
).agg(
    F.count("*").alias("total_goals"),
    F.countDistinct("match_id").alias("matches_played")
)

# STEP 2: Rank teams within each year
window_spec = Window.partitionBy("year").orderBy(F.desc("total_goals"))

year_top_team = year_team_goals.withColumn(
    "rank",
    F.row_number().over(window_spec)
)

# STEP 3: Filter only top team per year
year_top_team = year_top_team.filter(
    F.col("rank") == 1
).orderBy("year")

display(year_top_team)

### **2. Disciplined vs Most Aggressive Teams based on bookings**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# STEP 1: Aggregate metrics
bookings_gold_base = bookings_silver_df.groupBy(
    "team_id",
    "team_name",
    "team_code",
    "year"
).agg(
    F.sum("is_yellow_card").alias("yellow_cards"),
    F.sum("is_red_card").alias("red_cards"),
    F.sum("is_direct_red").alias("direct_red_cards"),
    F.sum("is_indirect_red").alias("indirect_red_cards"),
    F.count("*").alias("total_fouls"),
    F.countDistinct("match_id").alias("matches_played"),
    F.sum("is_early_booking").alias("early_fouls"),
    F.sum("is_late_booking").alias("late_fouls"),
    F.min("match_date").alias("first_match_date"),
    F.max("match_date").alias("last_match_date")
)

# STEP 2: Derived metrics
bookings_gold = bookings_gold_base.withColumn(
    "total_cards",
    F.col("yellow_cards") + F.col("red_cards")
).withColumn(
    "cards_per_match",
    F.round(F.col("total_cards") / F.col("matches_played"), 2)
)

# STEP 3: Aggression score
bookings_gold = bookings_gold.withColumn(
    "aggression_score",
    F.round(
        (F.col("yellow_cards") * 1) +
        (F.col("red_cards") * 3) +
        (F.col("early_fouls") * 1.5) +
        (F.col("cards_per_match") * 2),
        2
    )
)

# STEP 4: Classification
bookings_gold = bookings_gold.withColumn(
    "discipline_type",
    F.when(F.col("aggression_score") >= 15, "Highly Aggressive")
     .when(F.col("aggression_score") >= 8, "Moderate")
     .otherwise("Disciplined Team")
)

# STEP 5: Ranking
window_spec = Window.orderBy(F.desc("aggression_score"))

bookings_gold = bookings_gold.withColumn(
    "rank",
    F.dense_rank().over(window_spec)
)

# STEP 6: Best & Worst Teams
bookings_gold = bookings_gold.withColumn(
    "is_most_aggressive_team",
    F.when(F.col("rank") == 1, "Yes").otherwise("No")
)

# Optional: Most disciplined (lowest score)
window_low = Window.orderBy(F.asc("aggression_score"))

bookings_gold = bookings_gold.withColumn(
    "discipline_rank",
    F.dense_rank().over(window_low)
).withColumn(
    "is_most_disciplined_team",
    F.when(F.col("discipline_rank") == 1, "Yes").otherwise("No")
)

# FINAL OUTPUT
gold_bookings = bookings_gold.orderBy("rank");
display(gold_bookings)

In [0]:
def write(input_df: DataFrame):
    out_dir = f"{working_directory}/output/gold_bookings"
    
### Put your code here.
    mode_name = "overwrite"
    
    input_df. \
        write. \
        mode(mode_name). \
        parquet(out_dir)
    
    
write(gold_bookings)
dbutils.fs.ls(f"{working_directory}/output/gold_bookings")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# STEP 1: Aggregate base metrics
bookings_gold_base = bookings_silver_df.groupBy(
    "team_name",
    "team_code"
).agg(
    F.sum("is_yellow_card").alias("yellow_cards"),
    F.sum("is_red_card").alias("red_cards"),
    F.count("*").alias("total_fouls"),
    F.countDistinct("match_id").alias("matches_played"),
    F.sum("is_early_booking").alias("early_fouls"),
    F.sum("is_late_booking").alias("late_fouls")
)

# STEP 2: Derived metrics
bookings_gold = bookings_gold_base.withColumn(
    "total_cards",
    F.col("yellow_cards") + F.col("red_cards")
).withColumn(
    "cards_per_match",
    F.round(F.col("total_cards") / F.col("matches_played"), 2)
)

# STEP 3: Aggression Score (KEY KPI 🔥)
bookings_gold = bookings_gold.withColumn(
    "aggression_score",
    F.round(
        (F.col("yellow_cards") * 1) +     # light fouls
        (F.col("red_cards") * 3) +        # severe fouls
        (F.col("early_fouls") * 1.5) +    # aggressive start
        (F.col("cards_per_match") * 2),   # consistency
        2
    )
)

# STEP 4: Classification
bookings_gold = bookings_gold.withColumn(
    "discipline_type",
    F.when(F.col("aggression_score") >= 15, "Highly Aggressive")
     .when(F.col("aggression_score") >= 8, "Moderate")
     .otherwise("Disciplined Team")
)

# STEP 5: Ranking (Most Aggressive)
window_desc = Window.orderBy(F.desc("aggression_score"))

bookings_gold = bookings_gold.withColumn(
    "aggression_rank",
    F.dense_rank().over(window_desc)
)

# STEP 6: Ranking (Most Disciplined)
window_asc = Window.orderBy(F.asc("aggression_score"))

bookings_gold = bookings_gold.withColumn(
    "discipline_rank",
    F.dense_rank().over(window_asc)
)

# STEP 7: Flags
bookings_gold = bookings_gold.withColumn(
    "is_most_aggressive_team",
    F.when(F.col("aggression_rank") == 1, "Yes").otherwise("No")
).withColumn(
    "is_most_disciplined_team",
    F.when(F.col("discipline_rank") == 1, "Yes").otherwise("No")
)

# FINAL OUTPUT
display(bookings_gold.orderBy("aggression_rank"))

### **3. Most Tactical Teams (High Substitutions) and Substitution Timing(early/late)**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# STEP 1: Aggregate metrics
subs_gold_base = subs_silver_df.groupBy(
    "team_id",
    "team_name",
    "team_code",
    "year"
).agg(
    F.count("*").alias("total_substitutions"),
    F.countDistinct("match_id").alias("matches_played"),
    F.sum("is_early_sub").alias("early_substitutions"),
    F.sum("is_late_sub").alias("late_substitutions"),
    F.sum("is_player_in").alias("players_in"),
    F.sum("is_player_out").alias("players_out"),
    F.min("match_date").alias("first_match_date"),
    F.max("match_date").alias("last_match_date")
)

# STEP 2: Derived metrics
subs_gold = subs_gold_base.withColumn(
    "subs_per_match",
    F.round(F.col("total_substitutions") / F.col("matches_played"), 2)
)

# STEP 3: Tactical score
subs_gold = subs_gold.withColumn(
    "tactical_score",
    F.round(
        (F.col("early_substitutions") * 1.5) +   # aggressive strategy
        (F.col("late_substitutions") * 1.0) +    # defensive control
        (F.col("subs_per_match") * 2),           # overall activity
        2
    )
)

# STEP 4: Tactical classification
subs_gold = subs_gold.withColumn(
    "tactical_type",
    F.when(F.col("early_substitutions") > F.col("late_substitutions"), "Aggressive Manager")
     .when(F.col("late_substitutions") > F.col("early_substitutions"), "Defensive Manager")
     .otherwise("Balanced Strategy")
)

# STEP 5: Ranking (Best Tactical Team)
window_spec = Window.orderBy(F.desc("tactical_score"))

subs_gold = subs_gold.withColumn(
    "rank",
    F.dense_rank().over(window_spec)
)

subs_gold = subs_gold.withColumn(
    "is_best_tactical_team",
    F.when(F.col("rank") == 1, "Yes").otherwise("No")
)

# FINAL OUTPUT
gold_subs = subs_gold.orderBy("rank");
display(gold_subs)

In [0]:
def write(input_df: DataFrame):
    out_dir = f"{working_directory}/output/gold_subs"
    
### Put your code here.
    mode_name = "overwrite"
    
    input_df. \
        write. \
        mode(mode_name). \
        parquet(out_dir)
    
    
write(gold_subs)
dbutils.fs.ls(f"{working_directory}/output/gold_subs")

### **4. Classify Teams grade using Penalty**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# STEP 1: Aggregate metrics
penalty_gold_base = penalty_kicks_silver_df.groupBy(
    "team_id",
    "team_name",
    "team_code",
    "year"
).agg(
    F.count("*").alias("total_penalties"),
    F.sum("is_penalty_scored").alias("penalties_scored"),
    F.sum("is_penalty_missed").alias("penalties_missed"),
    F.countDistinct("match_id").alias("matches_played"),
    F.min("match_date").alias("first_match_date"),
    F.max("match_date").alias("last_match_date")
)

# STEP 2: Conversion rate
penalty_gold = penalty_gold_base.withColumn(
    "penalty_conversion_rate",
    F.round(
        F.col("penalties_scored") / F.col("total_penalties"),
        2
    )
)

# STEP 3: Penalties per match
penalty_gold = penalty_gold.withColumn(
    "penalties_per_match",
    F.round(F.col("total_penalties") / F.col("matches_played"), 2)
)

# STEP 4: Clinical score (key metric)
penalty_gold = penalty_gold.withColumn(
    "clinical_score",
    F.round(
        (F.col("penalty_conversion_rate") * 5) +   # finishing ability
        (F.col("penalties_per_match") * 2),        # attacking pressure
        2
    )
)

# STEP 5: Classification
penalty_gold = penalty_gold.withColumn(
    "clinical_type",
    F.when(F.col("penalty_conversion_rate") >= 0.85, "Highly Clinical")
     .when(F.col("penalty_conversion_rate") >= 0.65, "Average")
     .otherwise("Wasteful")
)

# STEP 6: Ranking
window_spec = Window.orderBy(F.desc("clinical_score"))

penalty_gold = penalty_gold.withColumn(
    "rank",
    F.dense_rank().over(window_spec)
)

# STEP 7: Best & Worst teams
penalty_gold = penalty_gold.withColumn(
    "is_best_clinical_team",
    F.when(F.col("rank") == 1, "Yes").otherwise("No")
)

# Worst (lowest conversion)
window_low = Window.orderBy(F.asc("penalty_conversion_rate"))

penalty_gold = penalty_gold.withColumn(
    "waste_rank",
    F.dense_rank().over(window_low)
).withColumn(
    "is_most_wasteful_team",
    F.when(F.col("waste_rank") == 1, "Yes").otherwise("No")
)

# FINAL OUTPUT
gold_penalty = penalty_gold.orderBy("rank")
display(gold_penalty)

In [0]:
def write(input_df: DataFrame):
    out_dir = f"{working_directory}/output/gold_penalty"
    
### Put your code here.
    mode_name = "overwrite"
    
    input_df. \
        write. \
        mode(mode_name). \
        parquet(out_dir)
    
    
write(gold_penalty)
dbutils.fs.ls(f"{working_directory}/output/gold_penalty")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# STEP 1: Aggregate metrics
penalty_gold_base = penalty_kicks_silver_df.groupBy(
    "team_name",
    "team_code"
).agg(
    F.count("*").alias("total_penalties"),
    F.sum("is_penalty_scored").alias("penalties_scored"),
    F.sum("is_penalty_missed").alias("penalties_missed"),
    F.countDistinct("match_id").alias("matches_played")
)

# STEP 2: Conversion rate
penalty_gold = penalty_gold_base.withColumn(
    "penalty_conversion_rate",
    F.round(
        F.col("penalties_scored") / F.col("total_penalties"),
        2
    )
)

# STEP 3: Penalties per match
penalty_gold = penalty_gold.withColumn(
    "penalties_per_match",
    F.round(
        F.col("total_penalties") / F.col("matches_played"), 2
    )
)

# STEP 4: Ranking (MOST penalties)
window_spec = Window.orderBy(F.desc("total_penalties"))

penalty_gold = penalty_gold.withColumn(
    "penalty_rank",
    F.dense_rank().over(window_spec)
)

# STEP 5: Flag best team
penalty_gold = penalty_gold.withColumn(
    "is_most_penalty_team",
    F.when(F.col("penalty_rank") == 1, "Yes").otherwise("No")
)

# FINAL OUTPUT
display(penalty_gold.orderBy("penalty_rank"))

**Catalogue**


1. Team Info
2. Goals (Attack Strength)
3. Substitutions (Tactical Behavior)
4. Bookings (Discipline)
5. Penalty (Clinical Finishing)

In [0]:
dbutils.widgets.text("catalog", "hive_metastore")
dbutils.widgets.text("schema", "team_analysis")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA_NAME = dbutils.widgets.get("schema")

if CATALOG != "hive_metastore":
    spark.sql(f"USE CATALOG {CATALOG}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_NAME}")

def write_to_catalog(df, table_name):
    full_table_name = f"{CATALOG}.{SCHEMA_NAME}.{table_name}"
    
    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(full_table_name)
    
    print(f"✅ Table written: {full_table_name}")


# write_to_catalog(goals_gold, "goals_gold")
# write_to_catalog(gold_subs, "gold_subs")
# write_to_catalog(gold_bookings, "gold_bookings")
# write_to_catalog(gold_penalty, "gold_penalty")

write_to_catalog(best_goals_gold, "best_goals_gold")
write_to_catalog(year_top_team, "year_top_team")
write_to_catalog(subs_gold, "substitutions_gold")
write_to_catalog(bookings_gold, "bookings_gold")
write_to_catalog(penalty_gold, "penalty_gold")

spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA_NAME}").show()


# Goals per team etc.
best_goals_gold.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fifa_worldcup.team_analysis.best_goals_gold")
year_top_team.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fifa_worldcup.team_analysis.year_top_team")

subs_gold.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fifa_worldcup.team_analysis.substitutions_per_team")

bookings_gold.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fifa_worldcup.team_analysis.bookings_gold")

penalty_gold.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fifa_worldcup.team_analysis.penalty_gold")